In [1]:
from epyt import epanet
import pandas as pd
import numpy as np
import datetime
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt

# Epanet

In [2]:
import chardet

def analisar_inp(caminho_arquivo):
    problemas = []

    # Detectar encoding do arquivo
    with open(caminho_arquivo, "rb") as f:
        raw = f.read()
        resultado = chardet.detect(raw)
        encoding_detectado = resultado['encoding']
        confianca = resultado['confidence']

    print(f"📌 Encoding detectado: {encoding_detectado} (confiança {confianca:.2f})")

    # Reabre com encoding detectado (ou força utf-8 se der erro)
    try:
        with open(caminho_arquivo, "r", encoding=encoding_detectado) as f:
            linhas = f.readlines()
    except:
        with open(caminho_arquivo, "r", encoding="utf-8", errors="replace") as f:
            linhas = f.readlines()

    # Percorre linha por linha
    for num, linha in enumerate(linhas, start=1):
        # Procura caracteres não ASCII
        for char in linha:
            if ord(char) > 127:  # fora do ASCII básico
                problemas.append((num, char, linha.strip()))
                break  # reporta só uma vez por linha

    if problemas:
        print("\n⚠️ Problemas encontrados (linhas com acentos/caracteres especiais):")
        for num, char, conteudo in problemas:
            print(f"  Linha {num}: caractere '{char}' → {conteudo}")
    else:
        print("\n✅ Nenhum caractere problemático encontrado.")

# Exemplo de uso
analisar_inp("Epanet\\Gama 2\\GAM2 V25.inp")


📌 Encoding detectado: ascii (confiança 1.00)

✅ Nenhum caractere problemático encontrado.


In [100]:
d = epanet('Epanet Gerados\\Gama2\\GAM2 V34.inp')

EPANET version 20200 loaded (EPyT version v1.2.1 - Last Update: 09/01/2024).
Input File GAM2 V34.inp loaded successfully.



In [106]:
# Plot links IDs
d.plot()


c:\Users\filipe_silva\AppData\Local\anaconda3\Lib\site-packages\epyt\epanet.py:14411: UserWarning: Error 102: no network data available
  warnings.warn(errmssg.value.decode())
c:\Users\filipe_silva\AppData\Local\anaconda3\Lib\site-packages\epyt\epanet.py:14411: UserWarning: Error 102: no network data available
  warnings.warn(errmssg.value.decode())
c:\Users\filipe_silva\AppData\Local\anaconda3\Lib\site-packages\epyt\epanet.py:14411: UserWarning: Error 102: no network data available
  warnings.warn(errmssg.value.decode())
c:\Users\filipe_silva\AppData\Local\anaconda3\Lib\site-packages\epyt\epanet.py:14411: UserWarning: Error 102: no network data available
  warnings.warn(errmssg.value.decode())


Exception: Not enough network nodes/links.

In [103]:
d

In [107]:
#Criando uma biblioteca para puxar os IDs corretos

ids_nós = {nome:index for nome, index in zip(d.getNodeNameID(), d.getNodeIndex())}
ids_nós

c:\Users\filipe_silva\AppData\Local\anaconda3\Lib\site-packages\epyt\epanet.py:14411: UserWarning: Error 102: no network data available
  warnings.warn(errmssg.value.decode())
c:\Users\filipe_silva\AppData\Local\anaconda3\Lib\site-packages\epyt\epanet.py:14411: UserWarning: Error 102: no network data available
  warnings.warn(errmssg.value.decode())


{}

In [ ]:
d.getNodeElevations(ids_nós["553"])

1129.219970703125

In [ ]:
nome_nó = d.getNodeNameID(ids_nós['553'])
nome_nó

'553'

In [ ]:
d.runsCompleteSimulation()

d.getNodePressure(ids_nós['553'])

33.77947998046875

In [ ]:
#Abrindo uma analise Hidraúlica para printar a pressão horária de um nó

d.openHydraulicAnalysis()
d.initializeHydraulicAnalysis()
tstep,P , T_H, D, H, F, S, = 1, [], [], [], [] ,[], []
tempo = []
dados = []
while (tstep>0):
    t = d.runHydraulicAnalysis()
    P.append(d.getNodePressure())
    D.append(d.getNodeActualDemand())
    H.append(d.getNodeHydraulicHead())
    S.append(d.getLinkStatus())
    F.append(d.getLinkFlows())
    T_H.append(t)
    tstep=d.nextHydraulicAnalysisStep()
    tempo.append(tstep)

    # dados.append([tstep, P, D, H, F, S])
d.closeHydraulicAnalysis()

dict_index_name = {index:nome for nome, index in zip(d.getNodeNameID(), d.getNodeIndex())}
dict_name_index = {nome:index for nome, index in zip(d.getNodeNameID(), d.getNodeIndex())}



for tempo, pressao in enumerate(P):
    tempo_formatado = str(datetime.timedelta(hours=tempo))
    print(tempo_formatado, pressao[dict_name_index["553"]-1])


0:00:00 33.779541015625
1:00:00 34.23301696777344
2:00:00 34.494476318359375
3:00:00 34.66709899902344
4:00:00 34.700904846191406
5:00:00 34.70051956176758
6:00:00 34.5302848815918
7:00:00 33.95145034790039
8:00:00 33.3693733215332
9:00:00 32.71510314941406
10:00:00 32.10373306274414
11:00:00 31.506317138671875
12:00:00 31.253080368041992
13:00:00 31.37836456298828
14:00:00 31.8070068359375
15:00:00 32.10072326660156
16:00:00 32.272796630859375
17:00:00 32.329681396484375
18:00:00 32.274234771728516
19:00:00 32.045955657958984
20:00:00 32.10454559326172
21:00:00 32.4423828125
22:00:00 32.868900299072266
23:00:00 33.27106857299805
1 day, 0:00:00 33.77948760986328


In [ ]:
if nome_nó in ids_nós:
    indice_nó = ids_nós[nome_nó]
    print(f"Pressão horária do nó {nome_nó}:")

Pressão horária do nó 553:


In [ ]:
d.getNodePressure(ids_nós['553'])

33.77940368652344

# Criação de arquivos Excel a partir do Epanet

In [6]:
import pandas as pd
list_ids = d.getNodeNameID()
eleva = d.getNodeElevations()

cordenada = d.getNodeCoordinates()
x = cordenada["x"]
y = cordenada["y"]

base =  d.getNodeBaseDemands()

teste = {
    "id":list_ids,
    "Elevação":eleva,
    "Demanda":list(base[1]),
    "X_COORD":x.values(),
    "Y_COORD":y.values(),

}
df_node = pd.DataFrame(teste)
df_node

,id,Elevação,Demanda,X_COORD,Y_COORD
0,2,906.349976,0.141303,204794.945,8239635.369
1,3,904.409973,0.188404,204874.115,8239649.583
2,4,923.229980,0.000000,204031.590,8238899.958
3,5,920.270020,0.061134,203989.215,8238939.168
4,6,943.200012,0.052408,202295.651,8240700.687
...,...,...,...,...,...
7079,228,916.549988,0.000000,203932.137,8238959.155
7080,232,916.390015,0.000000,203961.341,8239035.791
7081,233,916.390015,0.000000,203962.085,8239036.059
7082,RAP.SSB.001,1010.500000,0.000000,201312.895,8240231.900


In [4]:
d.getLinkInitialStatus

<bound method epanet.getLinkInitialStatus of <epyt.epanet.epanet object at 0x0000029B15F76420>>

In [5]:
status    = d.getLinkInitialStatus()  # 0 = CLOSED | 1 = OPEN
status

array([1., 1., 1., ..., 1., 1., 1.])

In [7]:
import pandas as pd

# Dicionário para mapear índice → nome do nó
dict_ids = {
    index: nome
    for nome, index in zip(d.getNodeNameID(), d.getNodeIndex())
}

# Nós inicial e final dos links
startnode = [dict_ids[ids[0]] for ids in d.getLinkNodesIndex()]
endnode   = [dict_ids[ids[1]] for ids in d.getLinkNodesIndex()]

# Propriedades dos links
roughness = d.getLinkRoughnessCoeff()
length    = d.getLinkLength()
diameter  = d.getLinkDiameter()
status    = d.getLinkInitialStatus() 

# Monta o DataFrame
df_link = pd.DataFrame({
    "id": d.getLinkNameID(),
    "startnode": startnode,
    "endnode": endnode,
    "roughness": roughness,
    "length": length,
    "diameter": diameter,
    "status": status
})

# Converte status numérico para texto
df_link["status"] = df_link["status"].map({
    0: "CLOSED",
    1: "OPEN"
})

df_link


,id,startnode,endnode,roughness,length,diameter,status
0,0,7867,7504,140.0,3.2711,110.0,OPEN
1,1,7664,7665,130.0,4.5834,250.0,OPEN
2,2,7663,7664,130.0,11.8994,250.0,OPEN
3,3,7662,7663,130.0,6.4606,250.0,OPEN
4,4,7661,7662,130.0,6.8711,250.0,OPEN
...,...,...,...,...,...,...,...
7644,1462,6562,6564,0.0,0.0000,100.0,OPEN
7645,1464,7915,7916,0.0,0.0000,100.0,OPEN
7646,1485,580,1286,0.0,0.0000,100.0,OPEN
7647,1490,2450,4335,0.0,0.0000,100.0,OPEN


In [140]:
df_link.dtypes

id            object
startnode     object
endnode       object
roughness    float64
length       float64
diameter     float64
status        object
dtype: object

In [7]:
df_link['status'].unique()


array(['OPEN', 'CLOSED'], dtype=object)

In [8]:
df_link[df_link['status']=='CLOSED']

,id,startnode,endnode,roughness,length,diameter,status
564,629,6487,6488,130.0,11.869100,180.0,CLOSED
625,697,6562,6564,137.0,6.999600,60.0,CLOSED
671,748,6522,2103,137.0,2.552800,60.0,CLOSED
685,767,6538,6539,120.0,14.496200,110.0,CLOSED
1016,1141,6012,6013,140.0,53.177299,63.0,CLOSED
1395,1532,7241,7234,120.0,7.772800,63.0,CLOSED
1433,1570,833,2587,140.0,6.999400,85.0,CLOSED
1463,1602,7258,3139,140.0,0.564400,63.0,CLOSED
1563,1712,8094,8095,140.0,2.149200,110.0,CLOSED
1667,1829,7540,7018,142.0,36.285099,180.0,CLOSED


In [9]:
lista_closed = df_link.loc[df_link['status'] == 'CLOSED', 'id'].tolist()


In [10]:
lista_closed

['629',
 '697',
 '748',
 '767',
 '1141',
 '1532',
 '1570',
 '1602',
 '1712',
 '1829',
 '2588',
 '2635',
 '2929',
 '3427',
 '3947',
 '3976',
 '4069',
 '4116',
 '4118',
 '4371',
 '4496',
 '4776',
 '4860',
 '5009',
 '5062',
 '5183',
 '5215',
 '5295',
 '5394',
 '5568',
 '5635',
 '5659',
 '5746',
 '5968',
 '5989',
 '6227',
 '6259',
 '6293',
 '6331',
 '6386',
 '6688',
 '7063',
 '7570',
 '7732',
 '7747',
 '7826',
 '8049',
 '8295',
 '8327',
 '8628',
 '8638']

# Tratamento com Redes

## Abrindo Arquivos

In [101]:
#Arquivo original que gerou o arquivo Epanet
rede = pd.read_excel("Tabelas para calibração\\Gama 2\\Rede completa GAM2.xlsx")
rede

,Unnamed: 0.1,ID,Unnamed: 0,codunidade,material,DIAMETRO,EXTENSAO,rugosidade,dataimplan,creationda,...,GerenciaMa,Validado,datatratada,Idade (anos),Diametro_f,Idade,Rugosidade1,CHW,NODENUM_INICIAL,NODENUM_FINAL
0,0,0,0,,PVC DEFOFO,200,19.200039,140.0,2023-06-22 00:00:00,2023-06-23,...,PASS,,2023-06-22,2,200,0,132,140.0,3506,7687
1,1,1,1,,PEAD,63,172.258617,140.0,2025-09-30 00:00:00,2025-10-22,...,PASS,,2025-09-30,0,100,0,130,140.0,8387,8388
2,2,2,2,,PEAD,63,4.215317,140.0,2025-09-30 00:00:00,2025-10-22,...,PASS,,2025-09-30,0,100,0,130,140.0,8389,8390
3,3,3,3,,PEAD,63,0.112595,140.0,2025-09-30 00:00:00,2025-10-22,...,PASS,,2025-09-30,0,100,0,130,140.0,8390,8391
4,4,4,4,,PEAD,63,0.171875,140.0,2025-09-30 00:00:00,2025-10-22,...,PASS,,2025-09-30,0,100,0,130,140.0,8391,8392
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8674,8681,8681,9136,AAT.GAM.050,FF,300,12.348777,125.0,1966-06-14 00:00:00,2020-07-30,...,PASS,,1966-06-14,59,300,60,58,58.0,6674,5060
8675,8682,8682,9137,,PVC,85,11.129870,132.5,1965-09-30 00:00:00,2013-10-11,...,PASS,sim,1965-09-30,60,100,60,50,132.5,4173,4174
8676,8683,8683,9138,,PVC,110,9.721968,132.5,1965-09-30 00:00:00,2014-12-19,...,PASS,sim,1965-09-30,60,100,60,50,132.5,4172,4173
8677,8684,8684,9139,AAT.GAM.050,FF,300,22.609982,125.0,1966-06-14 00:00:00,2015-02-27,...,PASS,,1966-06-14,59,300,60,58,58.0,95,96


In [4]:
# rede = rede.reset_index(drop=True).reset_index().rename(columns={"index": "ID"})
# rede

In [88]:
rede['codunidade'].unique()

array([' ', 'A.RED.GAM-D009', 'AAT.GAM.130', 'SAT.GAM.031', 'AAT.GAM.030',
       'AAT.GAM.090', 'A.RED.GAM-D007', 'SAT.GAM.032', 'SAT.GAM.033',
       'AAT.GAM.070', 'AAT.GAM.010', 'AAT.PTR.010', 'AAT.ALG.010',
       'AAT.GAM.050'], dtype=object)

In [6]:
# tirando_dmc = pd.read_excel('Tabelas para calibração\\Paranoá-Itapoã\\Nos tratado com perda estimada (Pelos DMCs).xlsx')
# tirando_dmc = tirando_dmc[['NODENUM','DMC_1','Pattern']]
# tirando_dmc

In [7]:
# rede = pd.merge(rede,tirando_dmc,left_on='NODENUM_INICIAL',right_on='NODENUM',how='inner')
# rede.columns

In [7]:
# #Redes que serão calibradas
# rede_calibra = pd.read_excel('Tabelas para calibração\\UDA.002\\rede_calibração - UDA 002 parte esquerda.xls')
# rede_calibra
# rede_calibra = rede[rede['UDA']=='UDA.GAM.003']

In [9]:
# #Correspondencia com a tabela original através das coordenadas para encontrar o ID da rede que foi para o Epanet
# df_correspondencia = pd.merge(rede,rede_calibra, 
#                               left_on=['X_INI', 'Y_INI', 'X_FIM', 'Y_FIM'],
#                               right_on=['Xinicial', 'Yinicial', 'Xfinal', 'Yfinal'], 
#                               how='inner')

In [10]:
# df_correspondencia

In [ ]:
# rede_calibra = rede[rede['NOME_FINAL']=='VRP.VCP.025']

In [ ]:
# rede = rede.reset_index()
# rede.rename(columns={'index': 'ID'}, inplace=True)

In [7]:
rede.columns

Index(['Unnamed: 0.1', 'ID', 'Unnamed: 0', 'codunidade', 'material',
       'DIAMETRO', 'EXTENSAO', 'rugosidade', 'dataimplan', 'creationda',
       'X_INI', 'Y_INI', 'X_FIM', 'Y_FIM', 'Sistema', 'Localidade', 'RAP',
       'UDA', 'DMC', 'BOOSTER', 'VRP', 'TAG_VAZAO', 'Zonapressa', 'ZonaManobr',
       'GerenciaMa', 'Validado', 'datatratada', 'Idade (anos)', 'Diametro_f',
       'Idade', 'Rugosidade1', 'CHW', 'NODENUM_INICIAL', 'NODENUM_FINAL'],
      dtype='object')

## Descobrindo redes fechadas

In [152]:
rede['ID'] = rede['ID'].astype(str)
a = pd.merge(rede,df_link,left_on='ID',right_on='id',how='left')
a = a[a['RAP_1_ini']=='RAP.SSB.002']
a

,Unnamed: 0.1,ID,Unnamed: 0,creationda,DIAMETRO,material,codunidade,rugosidade,dataimplan,EXTENSAO,...,Shape_Area_fim,RAP_1_fim,DMC_1_fim,id,startnode,endnode,roughness,length,diameter,status
0,0,0,0,2025-10-24,110,PEAD,,140.0,NaN,3.271127,...,409228.754154,RAP.SSB.002,DMC.SSB.009/10,0,7867,7504,140.0,3.271100,110.0,OPEN
1,1,1,1,2024-07-18,250,PVC DEFOFO,,135.0,00:00:00,4.583378,...,279193.954556,RAP.SSB.002,DMC.SSB.009/10,1,7664,7665,135.0,4.583400,250.0,OPEN
2,2,2,2,2024-07-18,250,PVC DEFOFO,,135.0,00:00:00,11.899364,...,279193.954556,RAP.SSB.002,DMC.SSB.009/10,2,7663,7664,135.0,11.899400,250.0,OPEN
3,3,3,3,2024-07-18,250,PVC DEFOFO,,135.0,00:00:00,6.460561,...,279193.954556,RAP.SSB.002,DMC.SSB.009/10,3,7662,7663,135.0,6.460600,250.0,OPEN
4,4,4,4,2024-07-18,250,PVC DEFOFO,,135.0,00:00:00,6.871102,...,279193.954556,RAP.SSB.002,DMC.SSB.009/10,4,7661,7662,135.0,6.871100,250.0,OPEN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8703,8703,8703,8703,2013-10-11,63,PEAD,,135.0,1980-01-12 00:00:00,45.335417,...,455088.144694,RAP.SSB.002,DMC.SSB.009/10,8703,572,440,135.0,45.335400,63.0,OPEN
8704,8704,8704,8704,2013-10-11,32,PEAD,,135.0,1980-01-12 00:00:00,162.067841,...,455088.144694,RAP.SSB.002,DMC.SSB.009/10,8704,572,1046,135.0,162.067795,32.0,OPEN
8705,8705,8705,8705,2014-08-22,63,PEAD,,135.0,1980-01-12 00:00:00,17.623585,...,138128.725721,RAP.SSB.002,DMC.SSB.009/10,8705,522,523,135.0,17.623600,63.0,OPEN
8706,8706,8706,8706,2014-08-22,63,PEAD,,135.0,1980-01-12 00:00:00,21.207171,...,138128.725721,RAP.SSB.002,DMC.SSB.009/10,8706,524,525,135.0,21.207199,63.0,OPEN


In [153]:
a_closed = a[a['ID'].isin(lista_closed)]
a_closed

,Unnamed: 0.1,ID,Unnamed: 0,creationda,DIAMETRO,material,codunidade,rugosidade,dataimplan,EXTENSAO,...,Shape_Area_fim,RAP_1_fim,DMC_1_fim,id,startnode,endnode,roughness,length,diameter,status
3567,3567,3567,3567,2024-07-18,250,PEAD,,137.5,1997-11-29 00:00:00,2.926121,...,279193.954556,RAP.SSB.002,DMC.SSB.009/10,3567,7656,3729,137.0,2.926100,250.0,CLOSED
3568,3568,3568,3568,2024-07-18,250,PEAD,,137.5,1997-11-29 00:00:00,2.926121,...,279193.954556,RAP.SSB.002,DMC.SSB.009/10,3568,6888,7656,137.0,2.926100,250.0,CLOSED
4891,4891,4891,4891,2013-10-11,110,PEAD,,135.0,1997-12-02 00:00:00,23.400077,...,230350.815161,RAP.SSB.002,DMC.SSB.011,4891,1400,2494,135.0,23.400101,110.0,CLOSED
6068,6068,6068,6068,2013-10-11,32,PEAD,,135.0,1997-12-02 00:00:00,73.940647,...,230350.815161,RAP.SSB.002,DMC.SSB.011,6068,3943,162,135.0,73.940598,32.0,CLOSED
6070,6070,6070,6070,2013-10-11,63,PEAD,,135.0,1997-12-02 00:00:00,26.460272,...,279193.954556,RAP.SSB.002,DMC.SSB.009/10,6070,3939,3936,135.0,26.460300,63.0,CLOSED
6092,6092,6092,6092,2013-10-11,63,PEAD,,135.0,1997-12-15 00:00:00,12.978744,...,230350.815161,RAP.SSB.002,DMC.SSB.011,6092,3956,3957,135.0,12.978700,63.0,CLOSED
6172,6172,6172,6172,2013-10-11,32,PEAD,,135.0,1997-11-29 00:00:00,2.848602,...,279193.954556,RAP.SSB.002,DMC.SSB.012,6172,3710,3687,135.0,2.848600,32.0,CLOSED


## Voltando ao normal

In [9]:
rede['VRP'].unique()
# rede['DMC_ini'].unique()
# rede['NOME'].unique()
# rede['UDA'].unique()
# rede['codunidade'].unique()

array(['VRP.GUA.009', ' ', 'VRP NB 02', 'VRP.CND.001', 'VRP.NBN.002',
       'VRP.GUA.019', 'VRP.GUA.020', 'VRP.GUA.014', 'VRP.CND.002',
       'VRP.NBN.001', 'VRP.CND.004', 'VRP.CND.003', 'VRP.NBN.027'],
      dtype=object)

In [ ]:
a = rede[rede['ID']==7610]
a

,Unnamed: 0.1,ID,Unnamed: 0,creationda,DIAMETRO,material,codunidade,rugosidade,dataimplan,EXTENSAO,...,Validado_fim,OBJECTID_fim,NOME_fim,SHAPE_Leng_fim,REGIÃO_fim,RA_fim,Shape_Le_1_fim,Shape_Area_fim,RAP_1_fim,DMC_1_fim
7610,7610,7610,7610,2024-10-25,150,FF,,125.0,1997-03-08 00:00:00,4.419084,...,,16719,VRP.SSB.012,7087.183586,NORTE,SÃO SEBASTIÃO,8481.995738,1.898251e+06,RAP.SSB.001,DMC.SSB.006/007


In [32]:
# rede = rede[rede['VRP']=="VRP.PRN.004"]
# rede = rede[rede['codunidade']=="SAT.NBN.014"]
# rede = rede[rede['UDA']=="UDA.RCE.002"]
# rede = rede[rede['NOME']=="VRP.SSB.013"]
# rede = rede[rede['DMC_1_ini']=="DMC.SSB.001"]
# rede = rede[rede['DMC_ini']=="DMC.SSB.007"]
rede

,Unnamed: 0.1,ID,Unnamed: 0_x,codunidade_x,material_x,DIAMETRO,EXTENSAO,rugosidade_x,dataimplan_x,creationda_x,...,POINT_M,OBJECTID,NOME,SHAPE_Leng,REGIÃO,RA,Shape_Le_1,Shape_Area,Longitude,Latitude
0,0,0,0,,PEAD,90,48.294146,140.0,2023-09-13 00:00:00,2023-12-01,...,NaN,0,VRP.SPW.010,0.000000,CENTRO,NÚCLEO BANDEIRANTES,11314.985684,2.536981e+06,-47.975852,-15.855665
1,1,1,1,AAT.NBN.010,FF,300,10.612114,125.0,00:00:00,2024-05-09,...,NaN,0,VRP.SPW.010,0.000000,CENTRO,NÚCLEO BANDEIRANTES,11314.985684,2.536981e+06,-47.973532,-15.869604
2,2,2,2,AAT.NBN.010,FF,300,10.107884,125.0,00:00:00,2024-05-09,...,NaN,0,VRP.SPW.010,0.000000,CENTRO,NÚCLEO BANDEIRANTES,11314.985684,2.536981e+06,-47.973485,-15.869689
3,3,3,3,,PEAD,63,40.654185,140.0,2023-10-31 00:00:00,2023-11-23,...,NaN,337,VRP.NBN.002,2597.863298,CENTRO,NÚCLEO BANDEIRANTES,2691.232192,3.471564e+05,-47.973732,-15.877058
4,4,4,4,,PEAD,63,70.939818,140.0,2023-10-31 00:00:00,2023-11-23,...,NaN,337,VRP.NBN.002,2597.863298,CENTRO,NÚCLEO BANDEIRANTES,2691.232192,3.471564e+05,-47.973771,-15.877423
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3207,3207,3207,3207,,PVC,60,44.554613,132.5,1972-11-18 00:00:00,2015-07-07,...,NaN,0,VRP.SPW.010,0.000000,CENTRO,NÚCLEO BANDEIRANTES,11314.985684,2.536981e+06,-47.970505,-15.870510
3208,3208,3208,3208,,PVC,60,38.982679,132.5,1972-11-18 00:00:00,2015-07-07,...,NaN,0,VRP.SPW.010,0.000000,CENTRO,NÚCLEO BANDEIRANTES,11314.985684,2.536981e+06,-47.973431,-15.872002
3209,3209,3209,3209,,PVC,85,8.425450,132.5,1972-11-18 00:00:00,2015-07-07,...,NaN,0,VRP.SPW.010,0.000000,CENTRO,NÚCLEO BANDEIRANTES,11314.985684,2.536981e+06,-47.973394,-15.872770
3210,3210,3210,3210,SAT.NBN.014,FF,500,20.154756,125.0,1972-09-29 00:00:00,2016-03-16,...,NaN,0,VRP.SPW.010,0.000000,CENTRO,NÚCLEO BANDEIRANTES,11314.985684,2.536981e+06,-47.970661,-15.866927


In [171]:
nos=pd.read_excel('Tabelas para calibração\\Nucleo Bandeirante Candagolandia\\Nos_parciais NBN CND.xlsx')

In [172]:
rede = pd.merge(rede,nos,left_on='NODENUM_INICIAL',right_on='NODENUM',how='left')
rede

,Unnamed: 0.1,ID,Unnamed: 0_x,codunidade_x,material_x,DIAMETRO,EXTENSAO,rugosidade_x,dataimplan_x,creationda_x,...,POINT_M,OBJECTID,NOME,SHAPE_Leng,REGIÃO,RA,Shape_Le_1,Shape_Area,Longitude,Latitude
0,0,0,0,,PEAD,90,48.294146,140.0,2023-09-13 00:00:00,2023-12-01,...,NaN,0,VRP.SPW.010,0.000000,CENTRO,NÚCLEO BANDEIRANTES,11314.985684,2.536981e+06,-47.975852,-15.855665
1,1,1,1,AAT.NBN.010,FF,300,10.612114,125.0,00:00:00,2024-05-09,...,NaN,0,VRP.SPW.010,0.000000,CENTRO,NÚCLEO BANDEIRANTES,11314.985684,2.536981e+06,-47.973532,-15.869604
2,2,2,2,AAT.NBN.010,FF,300,10.107884,125.0,00:00:00,2024-05-09,...,NaN,0,VRP.SPW.010,0.000000,CENTRO,NÚCLEO BANDEIRANTES,11314.985684,2.536981e+06,-47.973485,-15.869689
3,3,3,3,,PEAD,63,40.654185,140.0,2023-10-31 00:00:00,2023-11-23,...,NaN,337,VRP.NBN.002,2597.863298,CENTRO,NÚCLEO BANDEIRANTES,2691.232192,3.471564e+05,-47.973732,-15.877058
4,4,4,4,,PEAD,63,70.939818,140.0,2023-10-31 00:00:00,2023-11-23,...,NaN,337,VRP.NBN.002,2597.863298,CENTRO,NÚCLEO BANDEIRANTES,2691.232192,3.471564e+05,-47.973771,-15.877423
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3207,3207,3207,3207,,PVC,60,44.554613,132.5,1972-11-18 00:00:00,2015-07-07,...,NaN,0,VRP.SPW.010,0.000000,CENTRO,NÚCLEO BANDEIRANTES,11314.985684,2.536981e+06,-47.970505,-15.870510
3208,3208,3208,3208,,PVC,60,38.982679,132.5,1972-11-18 00:00:00,2015-07-07,...,NaN,0,VRP.SPW.010,0.000000,CENTRO,NÚCLEO BANDEIRANTES,11314.985684,2.536981e+06,-47.973431,-15.872002
3209,3209,3209,3209,,PVC,85,8.425450,132.5,1972-11-18 00:00:00,2015-07-07,...,NaN,0,VRP.SPW.010,0.000000,CENTRO,NÚCLEO BANDEIRANTES,11314.985684,2.536981e+06,-47.973394,-15.872770
3210,3210,3210,3210,SAT.NBN.014,FF,500,20.154756,125.0,1972-09-29 00:00:00,2016-03-16,...,NaN,0,VRP.SPW.010,0.000000,CENTRO,NÚCLEO BANDEIRANTES,11314.985684,2.536981e+06,-47.970661,-15.866927


In [94]:
rede['DMC_1_ini'].unique()

array(['DMC.SSB.001'], dtype=object)

In [102]:
rede['codunidade'].unique()

array([' ', 'A.RED.GAM-D009', 'AAT.GAM.130', 'SAT.GAM.031', 'AAT.GAM.030',
       'AAT.GAM.090', 'A.RED.GAM-D007', 'SAT.GAM.032', 'SAT.GAM.033',
       'AAT.GAM.070', 'AAT.GAM.010', 'AAT.PTR.010', 'AAT.ALG.010',
       'AAT.GAM.050'], dtype=object)

In [103]:
rede.columns

Index(['Unnamed: 0.1', 'ID', 'Unnamed: 0', 'codunidade', 'material',
       'DIAMETRO', 'EXTENSAO', 'rugosidade', 'dataimplan', 'creationda',
       'X_INI', 'Y_INI', 'X_FIM', 'Y_FIM', 'Sistema', 'Localidade', 'RAP',
       'UDA', 'DMC', 'BOOSTER', 'VRP', 'TAG_VAZAO', 'Zonapressa', 'ZonaManobr',
       'GerenciaMa', 'Validado', 'datatratada', 'Idade (anos)', 'Diametro_f',
       'Idade', 'Rugosidade1', 'CHW', 'NODENUM_INICIAL', 'NODENUM_FINAL'],
      dtype='object')

In [75]:
rede['TAG_VAZAO'].unique()

array(['VZ1.AAT.GAM.070', 'VZ1.AAT.GAM.070; VZ1.DMC.GAM.003',
       'VZ1.AAT.GAM.050', 'VZ1.AAT.GAM.090',
       'VZ1.AAT.GAM.070; VZ1.DMC.GAM.004'], dtype=object)

In [104]:
rede= rede[rede['Zonapressa']=='VRP.GAM.010']

In [76]:
rede=rede[rede['TAG_VAZAO']=='VZ1.AAT.GAM.050']

In [89]:
rede= rede[rede['codunidade']=='AAT.GAM.090']
rede

,Unnamed: 0.1,ID,Unnamed: 0,codunidade,material,DIAMETRO,EXTENSAO,rugosidade,dataimplan,creationda,...,GerenciaMa,Validado,datatratada,Idade (anos),Diametro_f,Idade,Rugosidade1,CHW,NODENUM_INICIAL,NODENUM_FINAL
734,736,736,914,AAT.GAM.090,FF,300,6.600061,125.0,00:00:00,2020-10-19,...,PASS,,2020-10-19,5,300,5,123,123.0,3655,2735
2105,2107,2107,2394,AAT.GAM.090,FF,300,2.053147,125.0,00:00:00,2014-08-06,...,PASS,,2014-08-06,11,300,10,112,112.0,2288,3655
2134,2136,2136,2429,AAT.GAM.090,FF,300,5.843314,125.0,00:00:00,2013-10-11,...,PASS,,2013-10-11,12,300,10,112,112.0,2288,2289
3699,3703,3703,4057,AAT.GAM.090,CA,300,132.718889,125.0,1988-12-12 00:00:00,2021-02-18,...,PASS,,1988-12-12,37,300,35,71,125.0,2884,986
3700,3704,3704,4058,AAT.GAM.090,CA,300,52.342230,125.0,1988-12-12 00:00:00,2021-02-18,...,PASS,,1988-12-12,37,300,35,71,125.0,1361,3948
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4997,5001,5001,5355,AAT.GAM.090,CA,300,16.165825,125.0,1988-12-12 00:00:00,2015-02-27,...,PASS,,1988-12-12,37,300,35,71,125.0,1032,1033
4999,5003,5003,5357,AAT.GAM.090,CA,300,19.929408,125.0,1988-12-12 00:00:00,2015-02-27,...,PASS,,1988-12-12,37,300,35,71,125.0,999,1000
8619,8626,8626,9081,AAT.GAM.090,FF,300,1.638496,125.0,1966-06-14 00:00:00,2020-10-19,...,PASS,,1966-06-14,59,300,60,58,58.0,2289,2298
8671,8678,8678,9133,AAT.GAM.090,FF,300,2.205111,125.0,1966-06-14 00:00:00,2013-10-11,...,PASS,,1966-06-14,59,300,60,58,58.0,2298,2290


In [95]:
rede.columns

Index(['Unnamed: 0.1', 'ID', 'Unnamed: 0_x', 'codunidade_x', 'material_x',
       'DIAMETRO', 'EXTENSAO', 'rugosidade_x', 'dataimplan_x', 'creationda_x',
       ...
       'POINT_M', 'OBJECTID', 'NOME', 'SHAPE_Leng', 'REGIÃO', 'RA',
       'Shape_Le_1', 'Shape_Area', 'Longitude', 'Latitude'],
      dtype='object', length=106)

In [122]:
rede['NOME'].unique()

array(['VRP.CND.001', 'VRP.CND.003', ' ', 'VRP.SPW.010'], dtype=object)

In [225]:
# Garantir que as colunas são strings
rede['Zonapressa'] = rede['Zonapressa'].astype(str)
rede['codunidade'] = rede['codunidade'].astype(str)

# Lista de zonas de pressão desejadas
zonas_desejadas = ['VRP.RCE.001']

# Criar o DataFrame filtrado e sobrescrever o original
rede = rede[rede['Zonapressa'].isin(zonas_desejadas) & (rede['codunidade'] == 'AAT.RCE.010')]

# Conferir resultado
print(f"Número de linhas no DataFrame filtrado: {len(rede)}")
print(rede.head())

Número de linhas no DataFrame filtrado: 88
      Unnamed: 0.1    ID  Unnamed: 0 creationda  DIAMETRO material  \
634            634   634         634 2024-06-17       200       FF   
1716          1716  1716        1716 2020-06-22       200       FF   
5383          5383  5383        5383 2017-06-06       400       FF   
5410          5410  5410        5410 2017-06-06       300       FF   
5411          5411  5411        5411 2017-06-06       300       FF   

       codunidade  rugosidade           dataimplan   EXTENSAO  ...  \
634   AAT.RCE.010       125.0             00:00:00   0.280146  ...   
1716  AAT.RCE.010       125.0             00:00:00   0.749120  ...   
5383  AAT.RCE.010       130.0  2013-10-11 00:00:00  20.047164  ...   
5410  AAT.RCE.010       130.0  2013-10-11 00:00:00  15.169208  ...   
5411  AAT.RCE.010       130.0  2013-10-11 00:00:00  19.333141  ...   

       Shape_Leng     Shape_Area  datatratada  Idade (anos) Diametro_f Idade  \
634   5865.739125  438786.832254   

In [123]:
rede['NOME'].unique()

array(['VRP.CND.001', 'VRP.CND.003', ' ', 'VRP.SPW.010'], dtype=object)

In [174]:
# Criando a lista de pontos desejados
pontos_filtrados = ["VRP.CND.001", "VRP.CND.002", "VRP.CND.003", "VRP.CND.004"]

# Filtrando a coluna 'NOME' que contém qualquer um dos valores da lista
rede = rede[rede['NOME'].isin(pontos_filtrados)]

rede

,Unnamed: 0.1,ID,Unnamed: 0_x,codunidade_x,material_x,DIAMETRO,EXTENSAO,rugosidade_x,dataimplan_x,creationda_x,...,POINT_M,OBJECTID,NOME,SHAPE_Leng,REGIÃO,RA,Shape_Le_1,Shape_Area,Longitude,Latitude
288,288,288,288,SAT.NBN.014,FF,300,41.809991,125.0,1991-12-27 00:00:00,2024-01-23,...,NaN,7365,VRP.CND.001,3613.28774,CENTRO,CANDANGOLÂNDIA,3613.28774,501159.614405,-47.952211,-15.855735
289,289,289,289,SAT.NBN.014,FF,300,18.699232,125.0,1991-12-27 00:00:00,2016-03-16,...,NaN,7365,VRP.CND.001,3613.28774,CENTRO,CANDANGOLÂNDIA,3613.28774,501159.614405,-47.950607,-15.853918
290,290,290,290,SAT.NBN.014,FF,300,13.180057,125.0,1991-12-27 00:00:00,2024-01-23,...,NaN,7365,VRP.CND.001,3613.28774,CENTRO,CANDANGOLÂNDIA,3613.28774,501159.614405,-47.951883,-15.855529
291,291,291,291,SAT.NBN.014,FF,300,40.998934,125.0,1991-12-27 00:00:00,2024-01-23,...,NaN,7365,VRP.CND.001,3613.28774,CENTRO,CANDANGOLÂNDIA,3613.28774,501159.614405,-47.951782,-15.855462
292,292,292,292,SAT.NBN.014,FF,300,13.585188,125.0,1991-12-27 00:00:00,2024-01-23,...,NaN,7365,VRP.CND.001,3613.28774,CENTRO,CANDANGOLÂNDIA,3613.28774,501159.614405,-47.951465,-15.855255
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1073,1073,1073,1073,SAT.NBN.014,FF,200,4.183235,125.0,1985-09-09 00:00:00,2024-05-08,...,NaN,7365,VRP.CND.001,3613.28774,CENTRO,CANDANGOLÂNDIA,3613.28774,501159.614405,-47.950238,-15.849654
1074,1074,1074,1074,SAT.NBN.014,FF,200,0.802516,125.0,1985-09-09 00:00:00,2024-05-08,...,NaN,7365,VRP.CND.001,3613.28774,CENTRO,CANDANGOLÂNDIA,3613.28774,501159.614405,-47.950246,-15.849656
1075,1075,1075,1075,SAT.NBN.014,FF,200,0.802516,125.0,1985-09-09 00:00:00,2024-05-08,...,NaN,7365,VRP.CND.001,3613.28774,CENTRO,CANDANGOLÂNDIA,3613.28774,501159.614405,-47.950253,-15.849658
1079,1079,1079,1079,SAT.NBN.014,FF,300,13.267714,125.0,1985-03-11 00:00:00,2024-01-26,...,NaN,7365,VRP.CND.001,3613.28774,CENTRO,CANDANGOLÂNDIA,3613.28774,501159.614405,-47.950736,-15.853384


In [43]:
rede = rede[rede['Zonapressa']=='VRP.GAM.003']
rede

,Unnamed: 0.1,ID,Unnamed: 0,codunidade,material,DIAMETRO,EXTENSAO,rugosidade,dataimplan,creationda,...,GerenciaMa,Validado,datatratada,Idade (anos),Diametro_f,Idade,Rugosidade1,CHW,NODENUM_INICIAL,NODENUM_FINAL
20,20,20,20,,PEAD,63,0.828578,140.0,2025-06-24 00:00:00,2025-07-21,...,PASS,sim,2025-06-24,0,100,0,130,140.0,8338,8352
21,21,21,21,,PEAD,110,5.999993,140.0,2025-06-24 00:00:00,2025-07-21,...,PASS,sim,2025-06-24,0,100,0,130,140.0,8353,8332
22,22,22,22,,PEAD,110,4.000017,140.0,2025-06-24 00:00:00,2025-07-21,...,PASS,sim,2025-06-24,0,100,0,130,140.0,8331,8353
23,23,23,23,,PEAD,110,0.669079,140.0,2025-06-24 00:00:00,2025-07-21,...,PASS,sim,2025-06-24,0,100,0,130,140.0,8337,8356
24,24,24,24,,PVC,60,1.905296,140.0,2025-06-24 00:00:00,2025-07-21,...,PASS,sim,2025-06-24,0,100,0,130,140.0,8356,8357
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7511,7515,7515,7869,,PVC,60,3.187776,132.5,1983-09-30 00:00:00,2017-07-19,...,PASS,sim,1983-09-30,42,100,40,57,132.5,1425,1652
7512,7516,7516,7870,,PVC,60,3.331827,132.5,1983-09-30 00:00:00,2017-07-19,...,PASS,sim,1983-09-30,42,100,40,57,132.5,1424,5989
7536,7540,7540,7894,,PVC,60,379.335574,132.5,1983-09-30 00:00:00,2017-07-19,...,PASS,sim,1983-09-30,42,100,40,57,132.5,5974,1812
7574,7578,7578,7932,,PVC,60,3.097877,132.5,1983-09-30 00:00:00,2017-07-19,...,PASS,sim,1983-09-30,42,100,40,57,132.5,3544,1952


In [105]:
lista_ids = rede['ID'].to_list()
lista_rugosidade = rede['CHW'].tolist()

In [ ]:
#Criando uma lista dos IDs da redes que serão calibradas
# lista_ids = rede[['ID','Diametro']].tolist()
# lista_ids = list(rede[['ID', 'rugosidade final']].itertuples(index=False, name=None))

# lista_ids[0:5]

[(611, 140.0), (779, 140.0), (856, 140.0), (898, 140.0), (967, 140.0)]

## Colocando rugosidade nova

In [11]:
# Criar o dicionário: {nome: índice}
dict_link_ids = {str(nome): index for nome, index in zip(d.getLinkNameID(), d.getLinkIndex())}

def aplicar_rugosidade_especifica(d, dict_link_ids, lista_ids, lista_rugosidade, 
                                  rugosidade_minima=50, rugosidade_maxima=150):
    """
    Atualiza a rugosidade dos trechos do modelo EPANET com base em uma lista
    de IDs e valores de rugosidade correspondentes.

    Parâmetros:
    - d: objeto do modelo EPANET
    - dict_link_ids: dicionário {nome_do_trecho: índice}
    - lista_ids: lista dos IDs dos trechos (ex: ['1', '2', '3'])
    - lista_rugosidade: lista com os novos valores de rugosidade (mesma ordem dos IDs)
    - rugosidade_minima: valor mínimo permitido
    - rugosidade_maxima: valor máximo permitido
    """
    
    if len(lista_ids) != len(lista_rugosidade):
        raise ValueError("❌ As listas 'lista_ids' e 'lista_rugosidade' devem ter o mesmo tamanho.")
    
    count_atualizados = 0

    for id_trecho, nova_rug in zip(lista_ids, lista_rugosidade):
        str_id = str(int(id_trecho))  # garante que o ID é string limpa
        nova_rug = float(nova_rug)

        if str_id in dict_link_ids:
            index = dict_link_ids[str_id]
            rug_atual = float(d.getLinkRoughnessCoeff(index))

            # Aplica limites
            nova_rug = max(rugosidade_minima, min(nova_rug, rugosidade_maxima))

            d.setLinkRoughnessCoeff(index, nova_rug)
            count_atualizados += 1

            print(f"✅ Trecho {str_id} (Índice {index}) - Rugosidade ajustada:")
            print(f"   {rug_atual:.2f} → {nova_rug:.2f}")
        else:
            print(f"⚠️ Trecho {str_id} não encontrado no modelo.")

    print(f"\n✅ Rugosidade ajustada em {count_atualizados} trechos de {len(lista_ids)}.")

# ======================
# 🧩 Exemplo de uso
# ======================

# Exemplo de listas (podem vir de um DataFrame)
# lista_ids = [1, 2, 3, 4]
# lista_rugosidade = [120, 110, 95, 80]

# Aplicar no modelo
aplicar_rugosidade_especifica(
    d, dict_link_ids, lista_ids, lista_rugosidade,
    rugosidade_minima=50, rugosidade_maxima=150
)


✅ Trecho 0 (Índice 1) - Rugosidade ajustada:
   130.00 → 137.00
✅ Trecho 1 (Índice 2) - Rugosidade ajustada:
   130.00 → 140.00
✅ Trecho 2 (Índice 3) - Rugosidade ajustada:
   130.00 → 140.00
✅ Trecho 3 (Índice 4) - Rugosidade ajustada:
   130.00 → 140.00
✅ Trecho 4 (Índice 5) - Rugosidade ajustada:
   130.00 → 140.00
✅ Trecho 5 (Índice 6) - Rugosidade ajustada:
   130.00 → 140.00
✅ Trecho 6 (Índice 7) - Rugosidade ajustada:
   130.00 → 140.00
✅ Trecho 7 (Índice 8) - Rugosidade ajustada:
   130.00 → 140.00
✅ Trecho 8 (Índice 9) - Rugosidade ajustada:
   130.00 → 140.00
✅ Trecho 9 (Índice 10) - Rugosidade ajustada:
   130.00 → 140.00
✅ Trecho 10 (Índice 11) - Rugosidade ajustada:
   130.00 → 137.50
✅ Trecho 11 (Índice 12) - Rugosidade ajustada:
   130.00 → 137.50
✅ Trecho 12 (Índice 13) - Rugosidade ajustada:
   130.00 → 137.50
✅ Trecho 13 (Índice 14) - Rugosidade ajustada:
   130.00 → 137.50
✅ Trecho 14 (Índice 15) - Rugosidade ajustada:
   130.00 → 137.50
✅ Trecho 15 (Índice 16) - Rug

In [12]:
d.saveInputFile('Epanet Gerados\\Agua Quente\\Agua Quente V7 reset rugosidade.inp')

## Criando as listas

In [262]:
rede.columns

Index(['Unnamed: 0.1', 'ID', 'Unnamed: 0', 'creationda', 'DIAMETRO',
       'material', 'codunidade', 'rugosidade', 'dataimplan', 'EXTENSAO',
       'X_INI', 'Y_INI', 'X_FIM', 'Y_FIM', 'Sistema', 'Localidade', 'RAP',
       'UDA', 'DMC', 'BOOSTER', 'VRP', 'TAG_VAZAO', 'Zonapressa', 'ZonaManobr',
       'GerenciaMa', 'Shape_Leng', 'Shape_Area', 'datatratada', 'Idade (anos)',
       'Diametro_f', 'Idade', 'Rugosidade1', 'CHW', 'NODENUM_INICIAL',
       'NODENUM_FINAL'],
      dtype='object')

In [106]:
# rede[['ID', 'Diametro']]

key = rede['ID'].to_list()
value = rede['CHW'].to_list()

dict_id_rugosidade = {k:v for k, v in zip(key, value)}


In [107]:
lista_ids = rede['ID'].to_list()

In [19]:
rede.columns

Index(['Unnamed: 0.1', 'ID', 'Unnamed: 0', 'codunidade', 'material',
       'DIAMETRO', 'EXTENSAO', 'rugosidade', 'dataimplan', 'creationda',
       'X_INI', 'Y_INI', 'X_FIM', 'Y_FIM', 'Sistema', 'Localidade', 'RAP',
       'UDA', 'DMC', 'BOOSTER', 'VRP', 'TAG_VAZAO', 'Zonapressa', 'ZonaManobr',
       'GerenciaMa', 'Validado', 'datatratada', 'Idade (anos)', 'Diametro_f',
       'Idade', 'Rugosidade1', 'CHW', 'NODENUM_INICIAL', 'NODENUM_FINAL'],
      dtype='object')

In [108]:
# Lista apenas por material

ids_por_material = rede.groupby('material')['ID'].apply(list).to_dict()
# ids_por_material

In [94]:
rede['material'].unique()

array(['FF', 'CA'], dtype=object)

In [16]:
# Lista material e diametro

def faixa_diametro(d):
    if d <= 125:
        return 'até_125'
    elif 125 < d < 550:
        return '125_a_550'
    elif 550 <= d < 1500:
        return '550_a_1500'
    else:
        return 'fora_faixa'  # opcional para pegar casos não previstos

rede['FAIXA_DIA'] = rede['DIAMETRO'].apply(faixa_diametro)

ids_por_grupo = rede.groupby(['MATERIAL', 'FAIXA_DIA'])['ID'].apply(list).to_dict()
print(ids_por_grupo)

{('FF', '125_a_550'): [320, 321, 322, 323, 324, 325, 327, 328, 329, 330, 331, 332, 333, 334, 339, 373, 374, 6473, 6480, 6483, 6511, 6512, 6513, 6521, 6560, 6632, 6637, 6638, 6640, 6647, 6651, 6652, 6655, 6656, 6658, 6663, 6664, 6670, 6696, 6697, 6699, 6715, 6734, 6735, 6736, 6775, 6798, 6819, 6820, 6825, 6826, 6827, 6828, 6829, 6830, 6831, 6837, 6847, 6912, 6913, 6919, 7040, 7044, 7264, 7265, 7266, 7267, 7268, 7269, 7272, 7273, 7274, 7277, 7278, 7279, 7280, 7281, 7283, 7284, 7285, 7286, 7287, 7468, 7469, 7470, 7496, 7510, 7511, 7512, 7564, 7634, 7646, 7717, 7723, 7724, 7725, 7726, 7727, 7730, 7735, 7938, 7940, 7941, 7954, 7955, 7961, 7962, 7964, 8208, 8233, 8251, 8309, 8322, 8334, 8365], ('FF', 'até_125'): [335, 336, 337, 6475, 6490, 6495, 6742, 6814, 6818, 6833, 6834, 6836, 6884, 7048, 7187, 7188, 7200, 7502, 7503, 7504, 7635, 7636, 7637, 7638, 7639, 7673, 7675, 7677, 7729, 7797, 7798, 7937, 7939, 7942, 7959, 7974, 7975, 7976, 8007, 8008, 8009, 8232, 8238, 8239, 8402, 8403, 8404, 8405

## Tratando Epanet

In [79]:
lista_ids[0]

(0, 63)

In [23]:
#Criando dicionário dos IDs das redes
dict_id_name_to_index = {nome:index for nome, index in zip(d.getLinkPipeNameID(), d.getLinkIndex())}
dict_id_name_to_index

{'0': 1,
 '1': 2,
 '2': 3,
 '3': 4,
 '4': 5,
 '5': 6,
 '6': 7,
 '7': 8,
 '8': 9,
 '9': 10,
 '10': 11,
 '11': 12,
 '12': 13,
 '13': 14,
 '14': 15,
 '15': 16,
 '16': 17,
 '17': 18,
 '18': 19,
 '19': 20,
 '20': 21,
 '21': 22,
 '22': 23,
 '23': 24,
 '24': 25,
 '25': 26,
 '26': 27,
 '27': 28,
 '28': 29,
 '29': 30,
 '30': 31,
 '31': 32,
 '32': 33,
 '33': 34,
 '34': 35,
 '35': 36,
 '36': 37,
 '37': 38,
 '38': 39,
 '39': 40,
 '40': 41,
 '41': 42,
 '42': 43,
 '43': 44,
 '44': 45,
 '45': 46,
 '46': 47,
 '47': 48,
 '48': 49,
 '49': 50,
 '50': 51,
 '51': 52,
 '52': 53,
 '53': 54,
 '54': 55,
 '55': 56,
 '56': 57,
 '57': 58,
 '58': 59,
 '59': 60,
 '60': 61,
 '61': 62,
 '62': 63,
 '63': 64,
 '64': 65,
 '65': 66,
 '66': 67,
 '67': 68,
 '68': 69,
 '69': 70,
 '70': 71,
 '71': 72,
 '72': 73,
 '73': 74,
 '74': 75,
 '75': 76,
 '76': 77,
 '77': 78,
 '78': 79,
 '79': 80,
 '80': 81,
 '81': 82,
 '82': 83,
 '83': 84,
 '84': 85,
 '85': 86,
 '86': 87,
 '87': 88,
 '88': 89,
 '89': 90,
 '90': 91,
 '91': 92,
 '92': 

In [81]:
dict_id_name_to_index[20]

21

In [ ]:
for id_name in dict_id_rugosidade:
    try:
        # print(dict_ids[id], dict_id_diametro[id])
        d.setLinkDiameter(dict_id_name_to_index[id_name], dict_id_rugosidade[id_name])
    except Exception as e:
        print(f"Erro no Id name: {e} ")

Erro no Id name: 355 
Erro no Id name: 356 
Erro no Id name: 887 
Erro no Id name: 993 
Erro no Id name: 994 
Erro no Id name: 1021 
Erro no Id name: 1022 
Erro no Id name: 1023 
Erro no Id name: 1024 
Erro no Id name: 1025 
Erro no Id name: 1041 
Erro no Id name: 1236 
Erro no Id name: 2553 
Erro no Id name: 2555 
Erro no Id name: 3947 
Erro no Id name: 5304 
Erro no Id name: 5544 
Erro no Id name: 5683 
Erro no Id name: 5693 
Erro no Id name: 5719 
Erro no Id name: 5732 
Erro no Id name: 6269 
Erro no Id name: 6386 
Erro no Id name: 6388 
Erro no Id name: 6407 
Erro no Id name: 7144 
Erro no Id name: 7157 
Erro no Id name: 7158 
Erro no Id name: 7159 
Erro no Id name: 7160 
Erro no Id name: 7176 
Erro no Id name: 7284 
Erro no Id name: 7289 
Erro no Id name: 7303 
Erro no Id name: 7360 
Erro no Id name: 7367 
Erro no Id name: 7378 
Erro no Id name: 7385 
Erro no Id name: 7386 
Erro no Id name: 7387 
Erro no Id name: 7388 
Erro no Id name: 7389 
Erro no Id name: 7402 
Erro no Id name:

In [83]:
d.saveInputFile('Epanet Gerados\\Vicente\\Vicente RV02')

In [62]:
# d.setLinkDiameter
d.getLinkDiameter(1)

array(63.)

In [13]:
dict_id_rugosidade = lista_ids


## Alterando a Rugosidade geral

In [243]:
# Criar o dicionário correto: {nome: index}
dict_link_ids = {nome: index for nome, index in zip(d.getLinkNameID(), d.getLinkIndex())}

def ajustar_rugosidade(d, dict_link_ids, lista_ids, ajuste=15, rugosidade_minima=40, rugosidade_maxima=150):
    count_item = 0

    for id in lista_ids:
        # Garantir que o id é uma string para acessar o dicionário
        str_id = str(id)

        if str_id in dict_link_ids:
            index = dict_link_ids[str_id]
            
            # Obter a rugosidade atual
            current_roughness = float(d.getLinkRoughnessCoeff(index))

            # Calcular o novo valor de rugosidade (com limites)
            novo_valor = current_roughness - ajuste
            novo_valor = max(rugosidade_minima, min(novo_valor, rugosidade_maxima))

            # Ajustar a rugosidade
            d.setLinkRoughnessCoeff(index, novo_valor)
            count_item += 1

            print(f"Nó {id} (Índice {index}):")
            print(f"  - Rugosidade: {current_roughness:.4f} → {novo_valor:.4f}")
        else:
            print(f"⚠️ ID {id} não encontrado no dicionário.")

    print(f"\n✅ Rugosidade ajustada em {count_item} nós.")

# Exemplo de uso:
ajustar_rugosidade(d, dict_link_ids, lista_ids, ajuste=20, rugosidade_minima=50, rugosidade_maxima=150)


Nó 3575 (Índice 3534):
  - Rugosidade: 140.0000 → 120.0000
Nó 3576 (Índice 3535):
  - Rugosidade: 140.0000 → 120.0000
Nó 3577 (Índice 3536):
  - Rugosidade: 140.0000 → 120.0000
Nó 3578 (Índice 3537):
  - Rugosidade: 140.0000 → 120.0000
Nó 4406 (Índice 4313):
  - Rugosidade: 140.0000 → 120.0000
Nó 4407 (Índice 4314):
  - Rugosidade: 140.0000 → 120.0000
Nó 4408 (Índice 4315):
  - Rugosidade: 140.0000 → 120.0000
Nó 4409 (Índice 4316):
  - Rugosidade: 140.0000 → 120.0000
Nó 4452 (Índice 4359):
  - Rugosidade: 140.0000 → 120.0000
Nó 4487 (Índice 4390):
  - Rugosidade: 140.0000 → 120.0000
Nó 4488 (Índice 4391):
  - Rugosidade: 140.0000 → 120.0000
Nó 5735 (Índice 5593):
  - Rugosidade: 137.5000 → 117.5000
Nó 5736 (Índice 5594):
  - Rugosidade: 137.5000 → 117.5000
Nó 5737 (Índice 5595):
  - Rugosidade: 137.5000 → 117.5000
Nó 5739 (Índice 5597):
  - Rugosidade: 137.5000 → 117.5000
Nó 5747 (Índice 5605):
  - Rugosidade: 137.5000 → 117.5000
Nó 6192 (Índice 5998):
  - Rugosidade: 140.0000 → 120.00

In [ ]:
d.getLinkLength(dict_link_ids['50'])

array(7.643188)

In [ ]:
d.setPattern()

In [ ]:
d.getLinkRoughnessCoeff(dict_link_ids['50'])

array(135.)

In [244]:
d.saveInputFile('Epanet Gerados\\Recanto_Riacho2\\Calibracao\\RCE_RF2_V39 um novo inicio.inp')

## Tentando ajuste por material e diametro

### Apenas por material

In [23]:
# ids_por_material

In [111]:
dict_link_ids = {nome: index for nome, index in zip(d.getLinkNameID(), d.getLinkIndex())}

def ajustar_rugosidade_por_listas_v2(
    d,
    dict_link_ids,
    ids_por_material,
    faixas_material,
    ajuste_geral
):
    total_ajustados = 0

    for material, lista_ids in ids_por_material.items():
        faixa = faixas_material.get(material, faixas_material['PADRÃO'])
        rug_min = faixa['min']
        rug_max = faixa['max']

        print(f"\n🔧 Ajustando material: {material} | Ajuste: {ajuste_geral} | Faixa: [{rug_min} - {rug_max}]")

        for id in lista_ids:
            str_id = str(id)

            if str_id in dict_link_ids:
                index = dict_link_ids[str_id]
                current_roughness = float(d.getLinkRoughnessCoeff(index))

                # Aplica o ajuste geral
                novo_valor = current_roughness - ajuste_geral

                # Respeita min e max da faixa
                novo_valor = max(min(novo_valor, rug_max), rug_min)

                d.setLinkRoughnessCoeff(index, novo_valor)
                total_ajustados += 1

                print(f"  ID {id} (Índice {index}): {current_roughness:.2f} → {novo_valor:.2f}")

            else:
                print(f"  ⚠️ ID {id} não encontrado no dicionário de índices!")

    print(f"\n✅ Rugosidade ajustada em {total_ajustados} links no total.")

# Um ajuste único para tudo
ajuste_geral = 10
# Faixas por material
faixas_material = {
    'PEAD': {'min': 120, 'max': 140},
    'PVC DEFOFO': {'min': 100, 'max': 135},
    'FF':   {'min': 55, 'max': 120},
    'PVC': {'min': 100, 'max': 135},
    'ACO': {'min': 50, 'max': 120},
    'PADRÃO': {'min': 100, 'max': 135}
}

# Roda!
ajustar_rugosidade_por_listas_v2(
    d,
    dict_link_ids,
    ids_por_material,
    faixas_material,
    ajuste_geral
)


🔧 Ajustando material: CA | Ajuste: 10 | Faixa: [100 - 135]
  ID 395 (Índice 394): 115.00 → 105.00
  ID 396 (Índice 395): 115.00 → 105.00
  ID 397 (Índice 396): 115.00 → 105.00
  ID 3577 (Índice 3510): 115.00 → 105.00
  ID 3589 (Índice 3522): 115.00 → 105.00
  ID 3603 (Índice 3536): 115.00 → 105.00
  ID 3604 (Índice 3537): 115.00 → 105.00
  ID 3605 (Índice 3538): 115.00 → 105.00
  ID 3607 (Índice 3540): 115.00 → 105.00
  ID 3617 (Índice 3550): 115.00 → 105.00
  ID 3634 (Índice 3564): 115.00 → 105.00
  ID 3671 (Índice 3600): 115.00 → 105.00
  ID 3705 (Índice 3634): 115.00 → 105.00
  ID 3706 (Índice 3635): 115.00 → 105.00
  ID 3797 (Índice 3723): 115.00 → 105.00
  ID 3798 (Índice 3724): 115.00 → 105.00
  ID 3799 (Índice 3725): 115.00 → 105.00
  ID 3800 (Índice 3726): 115.00 → 105.00
  ID 3806 (Índice 3732): 115.00 → 105.00
  ID 3832 (Índice 3757): 115.00 → 105.00
  ID 3839 (Índice 3764): 115.00 → 105.00
  ID 3840 (Índice 3765): 115.00 → 105.00
  ID 3845 (Índice 3770): 115.00 → 105.00
  I

In [112]:
d.saveInputFile('Epanet Gerados\\Gama2\\GAM2 V35.inp')

### Material e diametro

In [21]:
dict_link_ids = {nome: index for nome, index in zip(d.getLinkNameID(), d.getLinkIndex())}

def ajustar_rugosidade_por_grupo(
    d,
    dict_link_ids,
    ids_por_grupo,
    faixas_material_diametro,
    ajuste_geral
):
    total_ajustados = 0

    for (material, faixa_dia), lista_ids in ids_por_grupo.items():
        faixa = faixas_material_diametro.get((material, faixa_dia), faixas_material_diametro['PADRÃO'])
        rug_min = faixa['min']
        rug_max = faixa['max']

        print(f"\n🔧 Ajustando: Material={material} | Faixa Diametro={faixa_dia} | Redução={ajuste_geral} | Faixa: [{rug_min}-{rug_max}]")

        for id in lista_ids:
            str_id = str(id)

            if str_id in dict_link_ids:
                index = dict_link_ids[str_id]
                current_roughness = float(d.getLinkRoughnessCoeff(index))

                novo_valor = current_roughness - ajuste_geral  # sempre reduzir
                novo_valor = max(min(novo_valor, rug_max), rug_min)

                d.setLinkRoughnessCoeff(index, novo_valor)
                total_ajustados += 1

                print(f"  ID {id} (Índice {index}): {current_roughness:.2f} → {novo_valor:.2f}")

            else:
                print(f"  ⚠️ ID {id} não encontrado no dicionário!")

    print(f"\n✅ Rugosidade ajustada em {total_ajustados} links no total.")


faixas_material_diametro = {
    ('FOFO', 'até_125'): {'min': 80, 'max': 100},
    ('FOFO', '125_a_550'): {'min': 70, 'max': 90},
    ('FOFO', '550_a_1500'): {'min': 60, 'max': 80},

    ('PVC', 'até_125'): {'min': 120, 'max': 140},
    ('PVC', '125_a_550'): {'min': 110, 'max': 130},
    ('PVC', '550_a_1500'): {'min': 100, 'max': 120},

    ('AC', 'até_125'): {'min': 90, 'max': 110},
    ('AC', '125_a_550'): {'min': 80, 'max': 100},
    ('AC', '550_a_1500'): {'min': 70, 'max': 90},

    'PADRÃO': {'min': 60, 'max': 120}
}
    
ajuste_geral = -20  # significa reduzir 5 em todos

ajustar_rugosidade_por_grupo(
    d,
    dict_link_ids,
    ids_por_grupo,
    faixas_material_diametro,
    ajuste_geral
)




🔧 Ajustando: Material=FF | Faixa Diametro=até_125 | Redução=-20 | Faixa: [60-120]
  ID 887 (Índice 875): 110.00 → 120.00
  ID 891 (Índice 12223): 105.00 → 120.00
  ID 916 (Índice 903): 110.00 → 120.00
  ID 929 (Índice 916): 83.00 → 103.00
  ID 940 (Índice 927): 107.00 → 120.00

🔧 Ajustando: Material=PEAD | Faixa Diametro=125_a_550 | Redução=-20 | Faixa: [60-120]
  ID 592 (Índice 580): 107.00 → 120.00
  ID 593 (Índice 581): 107.00 → 120.00
  ID 610 (Índice 598): 107.00 → 120.00
  ID 619 (Índice 607): 107.00 → 120.00
  ID 634 (Índice 622): 107.00 → 120.00
  ID 764 (Índice 752): 107.00 → 120.00
  ID 813 (Índice 801): 107.00 → 120.00
  ID 816 (Índice 804): 107.00 → 120.00
  ID 909 (Índice 896): 107.00 → 120.00
  ID 913 (Índice 900): 107.00 → 120.00
  ID 919 (Índice 906): 107.00 → 120.00

🔧 Ajustando: Material=PEAD | Faixa Diametro=até_125 | Redução=-20 | Faixa: [60-120]
  ID 1695 (Índice 1673): 102.00 → 120.00
  ID 1696 (Índice 1674): 102.00 → 120.00
  ID 1697 (Índice 1675): 102.00 → 120.

## Acabou aqui

In [ ]:
#Rodando analise para avaliar se a troca de rugosidade influênciou no Nó
d.openHydraulicAnalysis()
d.initializeHydraulicAnalysis()
tstep,P , T_H, D, H, F, S, = 1, [], [], [], [] ,[], []
tempo = []
dados = []
while (tstep>0):
    t = d.runHydraulicAnalysis()
    P.append(d.getNodePressure())
    D.append(d.getNodeActualDemand())
    H.append(d.getNodeHydraulicHead())
    S.append(d.getLinkStatus())
    F.append(d.getLinkFlows())
    T_H.append(t)
    tstep=d.nextHydraulicAnalysisStep()
    tempo.append(tstep)

    # dados.append([tstep, P, D, H, F, S])
d.closeHydraulicAnalysis()

dict_index_name = {index:nome for nome, index in zip(d.getNodeNameID(), d.getNodeIndex())}
dict_name_index = {nome:index for nome, index in zip(d.getNodeNameID(), d.getNodeIndex())}



for tempo, pressao in enumerate(P):
    tempo_formatado = str(datetime.timedelta(hours=tempo))
    print(tempo_formatado, pressao[dict_name_index["PM.VRP.GAM.004"]-1])


0:00:00 56.342002868652344
1:00:00 57.00214385986328
2:00:00 57.48402404785156
3:00:00 57.74355697631836
4:00:00 57.86444091796875
5:00:00 57.82479476928711
6:00:00 57.57310104370117
7:00:00 56.68285369873047
8:00:00 55.78986740112305
9:00:00 54.836360931396484
10:00:00 53.76866912841797
11:00:00 52.93742370605469
12:00:00 52.32216262817383
13:00:00 52.412147521972656
14:00:00 53.022850036621094
15:00:00 53.44270324707031
16:00:00 53.76838684082031
17:00:00 53.92792892456055
18:00:00 53.92780685424805
19:00:00 53.688228607177734
20:00:00 53.8485221862793
21:00:00 54.23973083496094
22:00:00 54.69062805175781
23:00:00 55.395164489746094
1 day, 0:00:00 56.34178924560547


In [38]:
#Salvando o novo arquivo INP
d.saveInputFile('Epanet Gerados\\Vicente\\VCP RV00 -20.inp')

In [ ]:
len(lista_ids)

678

In [ ]:
count_item

676

In [ ]:
current_roughness

array(62.)

In [ ]:
d.getLinkRoughnessCoeff()

array([140., 140., 140., ...,   0.,   0.,   0.])